# Import libraries

In [ ]:

# Overriding read_screen_data to load from txt files instead of oscilloscope
def read_screen_data(scope, ip_address, num_of_cycles, wait_time, filename):
    import os
    import numpy as np
    import glob
    
    filepath = filename
    for d in ['task1', 'task2', 'task3', 'task4', 'task5', 'task6']:
        candidate = os.path.join(d, filename)
        if os.path.exists(candidate):
            filepath = candidate
            break
            
    print(f"Mocking read_screen_data: loading {filepath} instead of scope acquisition...")
    
    try:
        _loaded = np.loadtxt(filepath, skiprows=1)
        pts_per_cycle = len(_loaded) // num_of_cycles
        truncated_data = []
        for i in range(num_of_cycles):
            start = i * pts_per_cycle
            end = (i+1) * pts_per_cycle
            t = _loaded[start:end, 0]
            ch1 = _loaded[start:end, 1]
            ch2 = _loaded[start:end, 2]
            truncated_data.append([t, ch1, ch2])
        return scope, truncated_data
    except Exception as e:
        print(f"Failed to load {filepath}: {e}")
        return scope, []



: 

In [ ]:
import pyvisa
import numpy as np
import json
import time
import struct
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import gaussian_filter
from scipy.signal import find_peaks
from scipy.optimize import curve_fit

# Physical Constants for Scientific Rigor
h = 6.62607015e-34      # Planck constant (J*s)
e = 1.602176634e-19     # Elementary charge (C)
Phi0 = h / (2 * e)      # Magnetic flux quantum (Wb) approx 2.067e-15

# Global Plotting Configuration for Premium Aesthetics
plt.rcParams.update({
    'text.usetex': False,
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'figure.figsize': (10, 6),
    'figure.dpi': 100,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 1.5,
    'axes.titleweight': 'bold'
})


# Functions

In [ ]:
#Returns the voltage scaling in SI unit from the string read from the scope.

def parse_volt_per_div(s):
    """
    Owob scale string → volt/div (float)
    pl: '500mV->', '2V->', '200uV->'
    """
    s = s.replace('->', '').strip()

    if s.endswith('mV'):
        return float(s[:-2]) * 1e-3
    elif s.endswith('uV'):
        return float(s[:-2]) * 1e-6
    elif s.endswith('V'):
        return float(s[:-1])
    else:
        raise ValueError(f"Unknown voltage format: {s}")

#====================================================================================================
#====================================================================================================
#====================================================================================================

#Converts the datapoints into voltage values.

def adc_to_volt(adc, v_div, adc_bits=12):
    full_scale = v_div * 10 #10 total divs in voltage axis.
    lsb = full_scale / (2**adc_bits)
    
    adc_mid = np.mean(adc) #Removing offset.
    return (adc - adc_mid) * lsb

#====================================================================================================
#====================================================================================================
#====================================================================================================

#Returns the time scaling in SI unit from the string read from the scope.

def parse_time_per_div(s):
    s = s.replace('->', '').strip()

    if s.endswith('ns'):
        return float(s[:-2]) * 1e-9
    elif s.endswith('us'):
        return float(s[:-2]) * 1e-6
    elif s.endswith('ms'):
        return float(s[:-2]) * 1e-3
    elif s.endswith('s'):
        return float(s[:-1])
    else:
        raise ValueError(f"Unknown time format: {s}")

#====================================================================================================
#====================================================================================================
#====================================================================================================
        
#This bruteforce code stops the scope, reads the deepmemory data from CH1 and CH2 input. 
#The raw data is transformed the correct voltage and time values, which also returned and saved into a file.

#-----Inputs.-----
#scope: initialized scope name (rm.open_resource type, e.g.: "scope")
#ip_address: string of the scope IP address (e.g.: "TCPIP0::192.168.1.72::3000::SOCKET")
#num_of_cycles: The number of curves read from the scope.
#wait_time: The time in s to be waited between restarting (running) and stopping the scope.
#filename: string of the desired filename to save the data in .txt format (e.g.: "scope_data.txt").

#-----Outputs.-----
#scope: initialized scope name (rm.open_resource type, e.g.: "scope")
#truncated_list: 3D pyhton list, where 1st index is the one of the cycle, 2nd one is the coloumn (0=time, 1=ch1 voltage, 2=ch2 voltage), 3rd one is the datapoint

def read_deepmemory_data(scope, ip_address, num_of_cycles, wait_time, filename):
    results = []
    
    chunk_size = 40000
    timeout = 1000
    
    for cycle in np.arange(0,num_of_cycles):
        scope.write(":TRIGger:FORCe")
        time.sleep(0.1)
        scope.write(":RUNning STOP")
        time.sleep(0.1)
        scope.write(":ACQuire:DEPMEM 10K")
    
        #Read the scaling of the input channels and the time axis.
        scope.clear()
        ch1_scale_str = scope.query(":CH1:SCAle?")
        time.sleep(0.1)
        scope.clear()
        ch2_scale_str = scope.query(":CH2:SCAle?")
        time.sleep(0.1)
        scope.clear()
        time_scale_str = scope.query(":HORIzontal:SCAle?")
        time.sleep(0.1)
        
        scope.timeout = timeout #3 seconds timeout to read from deep memory.
        scope.chunk_size = chunk_size # 3.6 kByte buffer.
        
        #-----Read CH1 input.-----
        scope.write(":DATA:WAVE:DEPMem:CH1?")
        
        #Wait.
        time.sleep(0.1)
        
        #Read the raw data in chunks.
        raw_all_1 = b""
        while True:
            try:
                chunk_1 = scope.read_raw()
                raw_all_1 += chunk_1  
                #If the data is less than the buffer, we stop.
                if len(chunk_1) == 0:
                    break
            except pyvisa.errors.VisaIOError:
                #If there is timeout, there is no more data.
                break
        
        scope.close() #Stops the scope.
        
        #Reconnect the scope.
        rm = pyvisa.ResourceManager()
        scope = rm.open_resource(ip_address)
        
        scope.timeout = timeout
        scope.chunk_size = chunk_size
        
        scope.write_termination = '\r\n'
        scope.query_termination = '\r\n'
        scope.read_termination = '\r\n'
        
        scope.clear()
        scope.write("*CLS") #Clear the register.
        
        time.sleep(0.1)
    
        #-----Read CH2 input.-----
        scope.write(":DATA:WAVE:DEPMem:CH2?")
        
        #Wait.
        time.sleep(0.1)
        
        #Read the raw data in chunks.
        raw_all_2 = b""
        while True:
            try:
                chunk_2 = scope.read_raw()
                raw_all_2 += chunk_2  
                #If the data is less than the buffer, we stop.
                if len(chunk_2) == 0:
                    break
            except pyvisa.errors.VisaIOError:
                #If there is timeout, there is no more data.
                break        
        
        print(f"Total read bytes on CH1: {len(raw_all_1)}")
        print(f"Total read bytes on CH2: {len(raw_all_2)}")
        print(f"Cycle: {cycle}")
        
        scope.close() #Stops the scope.
    
        #-----Restart the scope, enable run mode.-----
    
        rm = pyvisa.ResourceManager()
        scope = rm.open_resource(ip_address)
        
        scope.timeout = timeout
        scope.chunk_size = chunk_size
        
        scope.write_termination = '\r\n'
        scope.query_termination = '\r\n'
        scope.read_termination = '\r\n'
        
        scope.clear()
        scope.write("*CLS") #Clear the register. 
        
        scope.write(":RUNning RUN") #Run the scope.
        
        time.sleep(0.1)
    
        #-----Data procession.-----
    
        #Cut the headers.
        cropped_raw_1 = raw_all_1[4:] 
        cropped_raw_2 = raw_all_2[4:] 
        
        #Calculate the number of datapoints.
        num_elements_1 = len(cropped_raw_1) // 2
        num_elements_2 = len(cropped_raw_2) // 2
    
        #Unpack the data by "struct.unpack".
        if num_elements_1 > 0:
            ch1_data = struct.unpack('<' + ('h' * num_elements_1), cropped_raw_1[:num_elements_1*2])
        else:
            print("Not enough data for CH1, try with higher voltage window.")
            
            scope.close() #Stops the scope.
    
            #-----Restart the scope, enable run mode.-----
    
            rm = pyvisa.ResourceManager()
            scope = rm.open_resource(ip_address)
        
            scope.timeout = 5000
        
            scope.write_termination = '\r\n'
            scope.query_termination = '\r\n'
            scope.read_termination = '\r\n'
        
            scope.clear()
            scope.write("*CLS") #Clear the register. 
        
            scope.write(":RUNning RUN") #Run the scope.
        
            time.sleep(0.1)
            empty=[]
            return scope, empty
        
        if num_elements_2 > 0:
            ch2_data = struct.unpack('<' + ('h' * num_elements_2), cropped_raw_2[:num_elements_2*2])
        else:
            print("Not enough data for CH2, try with higher voltage window.")
            scope.close() #Stops the scope.
    
            #-----Restart the scope, enable run mode.-----
    
            rm = pyvisa.ResourceManager()
            scope = rm.open_resource(ip_address)
        
            scope.timeout = timeout
        
            scope.write_termination = '\r\n'
            scope.query_termination = '\r\n'
            scope.read_termination = '\r\n'
        
            scope.clear()
            scope.write("*CLS") #Clear the register. 
        
            scope.write(":RUNning RUN") #Run the scope.
        
            time.sleep(0.1)
            return scope, empty
    
        #Check which list is longer, CH1 input or CH2 input data.
        min_length=np.min([len(ch1_data), len(ch2_data)])
        max_length=np.max([len(ch1_data), len(ch2_data)])
    
        #Recover voltage from raw data.
        ch1_volt = adc_to_volt(ch1_data, parse_volt_per_div(ch1_scale_str))[0:min_length]
        ch2_volt = adc_to_volt(ch2_data, parse_volt_per_div(ch2_scale_str))[0:min_length]
        
        #Recover time from raw data.
        sample_rate = min_length / (parse_time_per_div(time_scale_str) * (20)*(min_length/max_length)) #20 total divs in time axis.
        t = np.arange(min_length) / sample_rate #The time scaling is compensated with the unequal read values from CH1 and CH2 inputs.
    
        #Stack the data to n x 3 shape.
        #data_out = np.column_stack((t, ch1_volt, ch2_volt))
        
        #Save to file.
        # np.savetxt(
            # filename+"_"+str(cycle)+".txt",
            # data_out,
            # delimiter="\t",
            # header="time[s]\tCH1[V]\tCH2[V]",
            # comments=''
        # )
        results.append([t, ch1_volt, ch2_volt])
        time.sleep(wait_time)
        
    idx, shortest_list = min(enumerate(results[i][0] for i in range(len(results))), key=lambda x: len(x[1]))

    truncated_list = [
    [inner[:len(shortest_list)] for inner in middle] 
    for middle in results
    ]
    
    np_results = np.array(truncated_list)
    final_data = np_results.reshape(-1, 3)

    np.savetxt(
        filename,
        final_data,
        delimiter="\t",
        header="time[s]\tCH1[V]\tCH2[V] (num. of cycles: "+str(num_of_cycles)+")",
        comments=""
     )
    
    return scope, truncated_list

#====================================================================================================
#====================================================================================================
#====================================================================================================

#This bruteforce code stops the scope, reads the screen data from CH1 and CH2 input. 
#The raw data is transformed the correct voltage and time values, which also returned and saved into a file.

#-----Inputs.-----
#scope: initialized scope name (rm.open_resource type, e.g.: "scope")
#ip_address: string of the scope IP address (e.g.: "TCPIP0::192.168.1.72::3000::SOCKET")
#num_of_cycles: The number of curves read from the scope.
#wait_time: The time in s to be waited between restarting (running) and stopping the scope.
#filename: string of the desired filename to save the data in .txt format (e.g.: "scope_data.txt").

#-----Outputs.-----
#scope: initialized scope name (rm.open_resource type, e.g.: "scope")
#truncated_list: 3D pyhton list, where 1st index is the one of the cycle, 2nd one is the coloumn (0=time, 1=ch1 voltage, 2=ch2 voltage), 3rd one is the datapoint

def read_screen_data(scope, ip_address, num_of_cycles, wait_time, filename):
    results = []
    
    for i in np.arange(num_of_cycles):
        
        #Define timeout and chunk size (the length of the returned block in bytes).
        timeout=1000
        chunk_size=4+1520*2
        
        scope.write_termination = '\n'
        scope.read_termination = '\n'
        
        scope.write(":TRIGger:FORCe")
        time.sleep(0.1)
        scope.write(":RUNning STOP")
        time.sleep(0.1)
        
        scope.write(':DATA:WAVE:SCREen:HEAD?')
                
        #Wait.
        time.sleep(0.1)
                
        #Read the raw data in chunks.
        raw_all_noob = b""
        while True:
            try:
                chunk = scope.read_bytes(4+1520*2)
                raw_all_noob += chunk  
                #If the data is less than the buffer, we stop.
                if len(chunk) == 0:
                    del locals()[chunk]
                    break
            except pyvisa.errors.VisaIOError:
                        #If there is timeout, there is no more data.
                break
                
        scope.close() #Stops the scope.
        
        #Reconnect the scope.
        rm = pyvisa.ResourceManager()
        scope = rm.open_resource(ip_address)
                
        scope.timeout = timeout
        scope.chunk_size = chunk_size
                
        scope.write_termination = '\n'
        scope.query_termination = '\n'
        scope.read_termination = '\n'
                
        scope.clear()
        scope.write("*CLS") #Clear the register.
                
        
        scope.write(':DATA:WAVE:SCREen:CH1?')
                
        #Wait.
        time.sleep(0.1)
                
        #Read the raw data in chunks.
        raw_all_1 = b""
        while True:
            try:
                chunk = scope.read_bytes(4+1520*2)
                raw_all_1 += chunk  
                #If the data is less than the buffer, we stop.
                if len(chunk) == 0:
                    del locals()[chunk]
                    break
            except pyvisa.errors.VisaIOError:
                        #If there is timeout, there is no more data.
                break
                
        scope.close() #Stops the scope.
        
        #Reconnect the scope.
        rm = pyvisa.ResourceManager()
        scope = rm.open_resource(ip_address)
                
        scope.timeout = timeout
        scope.chunk_size = chunk_size
                
        scope.write_termination = '\r\n'
        scope.query_termination = '\r\n'
        scope.read_termination = '\r\n'
                
        scope.clear()
        scope.write("*CLS") #Clear the register.
        
        time.sleep(0.1)
        
        scope.write(':DATA:WAVE:SCREen:CH2?')
                
        #Wait.
        time.sleep(0.1)
                
        #Read the raw data in chunks.
        raw_all_2 = b""
        while True:
            try:
                chunk = scope.read_bytes(4+1520*2)
                raw_all_2 += chunk  
                #If the data is less than the buffer, we stop.
                if len(chunk) == 0:
                    del locals()[chunk]
                    break
            except pyvisa.errors.VisaIOError:
                        #If there is timeout, there is no more data.
                break
                
        scope.close() #Stops the scope.
        
        #Reconnect the scope.
        rm = pyvisa.ResourceManager()
        scope = rm.open_resource(ip_address)
                
        scope.timeout = timeout
        scope.chunk_size = chunk_size
                
        scope.write_termination = '\r\n'
        scope.query_termination = '\r\n'
        scope.read_termination = '\r\n'
                
        scope.clear()
        scope.write("*CLS") #Clear the register.

        #Get the screen voltage/time division info.
        ch1_scale_str = scope.query(":CH1:SCAle?")
        time.sleep(0.1)
        scope.clear()
        ch2_scale_str = scope.query(":CH2:SCAle?")
        time.sleep(0.1)
        scope.clear()
        time_scale_str = scope.query(":HORIzontal:SCAle?")
        time.sleep(0.1)
        
        scope.write(":RUNning RUN")
                
        time.sleep(0.1)
        
        print(f"Total read bytes on CH1: {len(raw_all_1)}")
        print(f"Total read bytes on CH1: {len(raw_all_2)}")
        print(f"Cycle: {i}")
        
        cropped_raw_1 = raw_all_1[4:] 
        cropped_raw_2 = raw_all_2[4:]
        num_elements_1 = len(cropped_raw_1) // 2
        num_elements_2 = len(cropped_raw_2) // 2
        ch1_data = struct.unpack('<' + ('h' * num_elements_1), cropped_raw_1[:num_elements_1*2])
        ch2_data = struct.unpack('<' + ('h' * num_elements_2), cropped_raw_2[:num_elements_2*2])
        min_length=np.min([len(ch1_data), len(ch2_data)])
        max_length=np.max([len(ch1_data), len(ch2_data)])
    
        #Recover voltage from raw data.
        ch1_volt = adc_to_volt(ch1_data, parse_volt_per_div(ch1_scale_str))[0:min_length]
        ch2_volt = adc_to_volt(ch2_data, parse_volt_per_div(ch2_scale_str))[0:min_length]
        
        #Recover time from raw data.
        sample_rate = min_length / (parse_time_per_div(time_scale_str) * (15.2)*(min_length/max_length)) #15.2 total divs in time axis.
        t = np.arange(min_length) / sample_rate #The time scaling is compensated with the unequal read values from CH1 and CH2 inputs.

        results.append([t, ch1_volt, ch2_volt])
        time.sleep(wait_time)
        
    idx, shortest_list = min(enumerate(results[i][0] for i in range(len(results))), key=lambda x: len(x[1]))

    truncated_list = [
    [inner[:len(shortest_list)] for inner in middle] 
    for middle in results
    ]
    
    np_results = np.array(truncated_list)
    final_data = np_results.reshape(-1, 3)

    np.savetxt(
        filename,
        final_data,
        delimiter="\t",
        header="time[s]\tCH1[V]\tCH2[V] (num. of cycles: "+str(num_of_cycles)+")",
        comments=""
     )
    
    return scope, truncated_list

#====================================================================================================
#====================================================================================================
#====================================================================================================

#1D Plotter function to show the read data in a twin-y axis figure.

def plotter_single(x, y1, y2):
    fig, ax1 = plt.subplots()

    #Set the colors.
    color1 = "tab:red"
    color2 = "tab:blue"

    #Left axis for y1.
    ax1.plot(x, y1, color=color1, label="SQUID signal")
    ax1.set_xlabel("t (s)", fontsize=12)
    ax1.set_ylabel("V_CH1 (V)", color=color1, fontsize=12)
    ax1.tick_params(axis="both", labelsize=10)
    ax1.tick_params(axis="y", labelcolor=color1)
    ax1.grid(True)

    #Right axis for y2.
    ax2 = ax1.twinx()
    ax2.plot(x, y2, color=color2, label="Reference")
    ax2.set_ylabel("V_CH2 (V)", color=color2, fontsize=12)
    ax2.tick_params(axis="y", labelcolor=color2, labelsize=10)

    #Set the legend.
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")

    plt.tight_layout()
    plt.show()

#====================================================================================================
#====================================================================================================
#====================================================================================================
    
#2D Plotter function to show a colormap of the measured data. 

def plotter_colormap(data_list, col_index=1, cmap='viridis'):
    
    data_array = np.array(data_list) #Converting the to a numpy array.
    
    slice_to_plot = data_array[:, col_index, :]# Slicing: row: first index, coloumn: third index. second index =1, which is channel 1 voltage
    
    plt.figure(figsize=(10, 6))
    img = plt.imshow(slice_to_plot, cmap=cmap, aspect='auto', origin='lower')
    
    plt.colorbar(img, label='3D color plot')
    plt.xlabel('t (s)')
    plt.ylabel('P (a.u.)')
    plt.title('3D color plot')

    plt.tight_layout()
    plt.show()

#====================================================================================================
#====================================================================================================
#====================================================================================================
    
#2D Plotter function to show a waterfalls plot of the data.

def plotter_waterfall(data_list, col_index=1, x_offset=0.05, y_offset=0.5):
    data_array = np.array(data_list)
    slice_to_plot = data_array[:, col_index, :]
    
    num_lines = slice_to_plot.shape[0]
    num_points = slice_to_plot.shape[1]

    #X axis.
    x = np.linspace(0, 10, num_points) 

    #Defining the colorscale.
    colors = ["red", "yellow", "green", "blue"]
    cmap = LinearSegmentedColormap.from_list("custom_wf", colors, N=num_lines)

    plt.figure()

    for i in range(num_lines):
        #Calculation of the offset for each curves.
        #i=num_lines-1 is blue, assuming a cooldown.
        current_x_offset = i * x_offset
        current_y_offset = i * y_offset
        
        #Data of the actual curve.
        display_x = x + current_x_offset
        display_y = slice_to_plot[i, :] + current_y_offset
        
        plt.plot(display_x, display_y, 
                 color=cmap(i / (num_lines - 1)), 
                 linewidth=1.5, 
                 alpha=0.8)

    plt.xlabel('t (s)')
    plt.ylabel('V_CH_1 (V) + offset')
    plt.title('V-I Cooldown')
    
    #Grid on.
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()

#====================================================================================================
#====================================================================================================
#====================================================================================================

#2D Plotter function to show a derivative, smoothed (Gauss filtered) plot of the measured data. 

from scipy.ndimage import gaussian_filter

def plotter_derivative_refined(data_list, col_index=1, cmap='RdBu_r', 
                               xlim=None, ylim=None, vlim=None, sigma=1):
    data_array = np.array(data_list)
    slice_to_plot = data_array[:, col_index, :]
    
    #Smooth to reduce noise.
    #Defining sigma for smoothing.
    if sigma > 0:
        slice_to_plot = gaussian_filter(slice_to_plot, sigma=sigma)
    
    #Derivative of the curve along the x axis.
    derivative_slice = np.gradient(slice_to_plot, axis=1)
    
    plt.figure(figsize=(10, 6))
    
    #Colormap limits.
    if vlim:
        vmin, vmax = vlim
    else:
        #Calculation of a sensible limit depending on the noise.
        limit = np.nanpercentile(np.abs(derivative_slice), 95)
        vmin, vmax = -limit, limit

    img = plt.imshow(derivative_slice, cmap=cmap, aspect='auto', 
                     origin='lower', vmin=vmin, vmax=vmax)
    
    plt.colorbar(img, label='dV/dI (a.u.)')
    if xlim: plt.xlim(xlim)
    if ylim: plt.ylim(ylim)
        
    plt.xlabel('t (s)')
    plt.ylabel('P (a.u.)')
    
    plt.title(f'Smoothed Shapiro (sigma={sigma})')
    plt.show()

In [ ]:
# Advanced Analysis Functions for Shapiro Steps and Rigor

def calculate_theoretical_v_step(frequency_ghz):
    """Calculates the theoretical Shapiro voltage step V = h*f / 2e."""
    f = frequency_ghz * 1e9
    v_step = (h * f) / (2 * e)
    return v_step

def detect_shapiro_steps(v, reference_v, freq_ghz, sigma=2):
    """
    Identifies Shapiro steps by finding peaks in dV/dI (derivative of voltage wrt bias).
    """
    # Smooth data
    v_smooth = gaussian_filter(v, sigma=sigma)
    dv = np.gradient(v_smooth)
    
    # Peaks in derivative correspond to transition regions (slopes of steps)
    # Valleys in derivative correspond to flat steps
    # Better: Plot dV/dI to see the peaks clearly.
    return dv

def fit_flux_quantum(frequencies, v_steps):
    """
    Fits V_step = Phi0 * f to find Phi0.
    Phi0 = h/2e 
    """
    def linear_model(f, p0):
        return p0 * f
    
    # Convert frequencies to Hz
    f_hz = np.array(frequencies) * 1e9
    popt, pcov = curve_fit(linear_model, f_hz, v_steps)
    return popt[0], pcov


# Open & check communication

In [ ]:
rm = pyvisa.ResourceManager()
print(rm)

ip_address="TCPIP0::192.168.1.72::3000::SOCKET" #Check the IP address and the port (192.168.1.x::YYYY). Set static IP on the PC if it is not.

scope = rm.open_resource(ip_address) #Open communication.

scope.timeout = 500 #Set timeout in ms. Recommended is >5000 for readout.

scope.write_termination = '\r\n' #Set terminator commands.
scope.query_termination = '\r\n'
scope.read_termination = '\r\n'

print(scope) #Print the type of the scope.

time.sleep(0.1)

print(scope.query("*IDN?")) #Check communication with "*IDN? and print."

time.sleep(0.1)

scope.write("*CLS") #Clear the register.
#scope.write("*RST") #Restart. Uncomment if needed.
scope.clear() #Clear buffer and wait.
time.sleep(1.0)

# ---START WORKING HERE---

In [ ]:
num_of_cycles=5
scope, truncated_data = read_screen_data(scope, ip_address, num_of_cycles, wait_time=10, filename="5.txt")
truncated_data = np.array(truncated_data)

In [ ]:
for i in range(num_of_cycles):
    t = truncated_data[i][0]
    ch1 = truncated_data[i][1]
    ch2 = truncated_data[i][2]
    plotter_single(t, ch1, ch2)

In [ ]:
# sweep down
num_of_cycles=10
scope, truncated_data = read_screen_data(scope, ip_address, num_of_cycles, wait_time=15, filename="down.txt")
truncated_data = np.array(truncated_data)

for i in range(num_of_cycles):
    t = truncated_data[i][0]
    ch1 = truncated_data[i][1]
    ch2 = truncated_data[i][2]
    plotter_single(t, ch1, ch2)

In [ ]:
# sweep down
num_of_cycles=4
scope, truncated_data = read_screen_data(scope, ip_address, num_of_cycles, wait_time=20, filename="down2.txt")
truncated_data = np.array(truncated_data)

for i in range(num_of_cycles):
    t = truncated_data[i][0]
    ch1 = truncated_data[i][1]
    ch2 = truncated_data[i][2]
    plotter_single(t, ch1, ch2)

In [ ]:
# magnet
num_of_cycles=5
scope, truncated_data = read_screen_data(scope, ip_address, num_of_cycles, wait_time=1, filename="mag.txt")
truncated_data = np.array(truncated_data)

for i in range(num_of_cycles):
    t = truncated_data[i][0]
    ch1 = truncated_data[i][1]
    ch2 = truncated_data[i][2]
    plotter_single(t, ch1, ch2)

In [ ]:
# magnet
num_of_cycles=5
scope, truncated_data = read_screen_data(scope, ip_address, num_of_cycles, wait_time=1, filename="mag2.txt")
truncated_data = np.array(truncated_data)

for i in range(num_of_cycles):
    t = truncated_data[i][0]
    ch1 = truncated_data[i][1]
    ch2 = truncated_data[i][2]
    plotter_single(t, ch1, ch2)

In [ ]:
# no magnet
num_of_cycles=1
scope, truncated_data = read_screen_data(scope, ip_address, num_of_cycles, wait_time=1, filename="no_mag.txt")
truncated_data = np.array(truncated_data)

for i in range(num_of_cycles):
    t = truncated_data[i][0]
    ch1 = truncated_data[i][1]
    ch2 = truncated_data[i][2]
    plotter_single(t, ch1, ch2)

In [ ]:
#task 2
num_of_cycles=1
scope, truncated_data = read_screen_data(scope, ip_address, num_of_cycles, wait_time=1, filename="task2_41hz_drive_1-0dc.txt")
truncated_data = np.array(truncated_data)

for i in range(num_of_cycles):
    t = truncated_data[i][0]
    ch1 = truncated_data[i][1]
    ch2 = truncated_data[i][2]
    plotter_single(t, ch1, ch2)

In [ ]:
### we want to find out the periodicity of the flux wrt the applied voltage
# the freq of the V was 41hz, in 2 periods of the V signal, there are aprox 14 periods of the cos(pi*flux/flux_0)
# the amplitude pk-pk amplitude of the V is 6V
# the flux is periodic wrt to the V, and the period is 0.82V , in current, this is 82 microA

# Task 3: Determination of Mutual Inductance $M$

### Theoretical Derivation
The SQUID consists of two Josephson junctions in a superconducting loop. The total critical current $I_c$ is modulated by the magnetic flux $\Phi_{ext}$ according to:
$$I_c(\Phi_{ext}) = 2I_0 \left| \cos\left(\frac{\pi \Phi_{ext}}{\Phi_0}\right) \right|$$
where $\Phi_0 = \frac{h}{2e} \approx 2.0678 \times 10^{-15} \text{ Wb}$ is the magnetic flux quantum.

The external flux is coupled via a flux bias line carrying current $I_{flux}$:
$$\Phi_{ext} = M \cdot I_{flux} + \Phi_{env}$$
where $M$ is the mutual inductance between the flux line and the SQUID loop, and $\Phi_{env}$ is the background magnetic flux.

The SQUID voltage response $V_{sq}$ follows the periodicity of $I_c$. The response is periodic in $\Phi_{ext}$ with a period of $\Phi_0$. Consequently, it is periodic in $I_{flux}$ with a period $\Delta I_{flux}$:
$$M \cdot \Delta I_{flux} = \Phi_0 \implies M = \frac{\Phi_0}{\Delta I_{flux}}$$

In the experimental setup, $I_{flux}$ is generated by applying a voltage $V_{ref}$ across a bias resistor $R_{bias} = 10 \text{ k}\Omega$:
$$I_{flux} = \frac{V_{ref}}{R_{bias}}$$
The period observed in terms of the reference voltage $\Delta V_{ref}$ is:
$$\Delta V_{ref} = \Delta I_{flux} \cdot R_{bias}$$

Therefore, the mutual inductance is determined as:
$$M = \frac{\Phi_0 \cdot R_{bias}}{\Delta V_{ref}}$$

In [ ]:
# Analysis of Mutual Inductance M
from scipy.signal import find_peaks
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
import os

# Constants
PHI_0 = 2.06783383e-15  # Magnetic flux quantum [Wb]
R_BIAS = 10000          # Bias resistor [Ohm]

# --- DATA SELECTION ---
# Available files: mag.txt, mag2.txt, task2_41hz_drive_1-5dc.txt, etc.
DEFAULT_FILE = "mag.txt"

v_sq = None
v_ref = None

if 'data' in locals() and data is not None:
    print("Using data from live measurement memory.")
    v_sq = np.array(data[0][1])
    v_ref = np.array(data[0][2])
else:
    if os.path.exists(DEFAULT_FILE):
        print(f"'data' variable not found. Loading from file: {DEFAULT_FILE}")
        try:
            # skipping the first header line
            loaded_data = np.loadtxt(DEFAULT_FILE, skiprows=1)
            v_sq = loaded_data[:, 1]
            v_ref = loaded_data[:, 2]
        except Exception as e:
            print(f"Error loading file: {e}")
    else:
        print(f"Error: No live 'data' and '{DEFAULT_FILE}' not found.")
        print("Please run the measurement cell or check if your data file exists.")

if v_sq is not None:
    # 1. Smoothing to improve peak detection
    sigma_val = 10 # Adjust based on noise
    v_sq_smooth = gaussian_filter(v_sq, sigma=sigma_val)
    
    # 2. Peak Detection
    # We search for peaks in the SQUID signal
    # prominence should be set higher than the noise floor
    peaks, _ = find_peaks(v_sq_smooth, prominence=0.005) 
    
    if len(peaks) > 1:
        # 3. Calculate periodicity Delta V_ref
        # Delta V_ref is the change in bias voltage required for one flux quantum change
        delta_v_ref_list = np.abs(np.diff(v_ref[peaks]))
        avg_delta_v_ref = np.mean(delta_v_ref_list)
        
        # 4. Calculate M
        # M = PHI_0 / Delta I_flux = PHI_0 / (Delta V_ref / R_BIAS)
        m_henry = (PHI_0 * R_BIAS) / avg_delta_v_ref
        
        # 5. Plotting
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 10), sharex=True)
        
        # Plot vs time to see the raw signal stability
        ax1.plot(v_sq, color='lightgray', alpha=0.5, label='SQUID Raw')
        ax1.plot(v_sq_smooth, color='blue', label=f'SQUID Smoothed (sigma={sigma_val})')
        ax1.plot(peaks, v_sq_smooth[peaks], "rx", label='Detected Peaks')
        ax1.set_ylabel('SQUID Voltage $V_{sq}$ (V)')
        ax1.legend()
        ax1.grid(True)
        
        # Plot Interference Pattern (V_sq vs V_ref)
        ax2.plot(v_ref, v_sq, 'k.', markersize=0.5, alpha=0.3, label='Data pts')
        ax2.plot(v_ref[peaks], v_sq_smooth[peaks], 'ro', label='Peak points')
        ax2.set_xlabel('Flux Bias Voltage $V_{ref}$ (V)')
        ax2.set_ylabel('SQUID Voltage $V_{sq}$ (V)')
        ax2.legend()
        ax2.grid(True)
        
        plt.suptitle('Task 3: Mutual Inductance Analysis')
        plt.tight_layout()
        plt.show()
        
        print(f"\n--- Task 3 Results ---")
        print(f"Number of periods analyzed: {len(delta_v_ref_list)}")
        print(f"Avg. Bias Voltage Period (Delta V_ref): {avg_delta_v_ref:.6f} V")
        print(f"Avg. Bias Current Period (Delta I_flux): {avg_delta_v_ref/R_BIAS*1e6:.3f} uA")
        print(f"Calculated Mutual Inductance M: {m_henry:.4e} H")
        print(f"Calculated Mutual Inductance M: {m_henry * 1e9:.4f} nH")
    else:
        print(f"Error: Only {len(peaks)} peaks detected. Need at least 2.")
        print("Try decreasing 'prominence' or 'sigma' in the code cell.")


In [ ]:
#task 3
# we know the flux period. we measure 10 times within a period. we do 80mv jumps
# save the files and  then data analysis at home

# there are some instabilities in the measurement and the plateaus are changing in time, even when the applied currents are kept the same. 
# in order to be consistent, we measure the size of the pleateau when they are the largest

num_of_cycles=1
scope, truncated_data = read_screen_data(scope, ip_address, num_of_cycles, wait_time=1, filename="task3_10.txt")
truncated_data = np.array(truncated_data)

# we mesaured 10 measurements within a period. we observed a small modulation as we varied flux. 
# however, the error of our measurement is quite high as the time fluctiation of the system is comparable (in magnitude) with the modulation
# thus it is hard to accurately measure the modulation when the system is fluctuating in time

# Task 4: Detection of Shapiro effect at 10 GHz

In [ ]:
filename = "task4_shapiro_11-5ghz.txt"
scope, data = read_screen_data(scope, ip_address, 1, 0.1, filename)
t, v_sq, v_ref = data[0]
plotter_single(t, v_sq, v_ref)


In [ ]:
# Task 5: Frequency Dependence of Shapiro Steps

In [ ]:
filename = "task4_shapiro_8ghz.txt"
scope, data = read_screen_data(scope, ip_address, 1, 0.1, filename)
t, v_sq, v_ref = data[0]
plotter_single(t, v_sq, v_ref)


In [ ]:

filename = "task4_shapiro_9ghz.txt"
scope, data = read_screen_data(scope, ip_address, 1, 0.1, filename)
t, v_sq, v_ref = data[0]
plotter_single(t, v_sq, v_ref)


In [ ]:
filename = "task4_shapiro_9.5ghz.txt"
scope, data = read_screen_data(scope, ip_address, 1, 0.1, filename)
t, v_sq, v_ref = data[0]
plotter_single(t, v_sq, v_ref)


In [ ]:
filename = "task4_shapiro_10ghz.txt"
scope, data = read_screen_data(scope, ip_address, 1, 0.1, filename)
t, v_sq, v_ref = data[0]
plotter_single(t, v_sq, v_ref)


In [ ]:
filename = "task4_shapiro_11ghz.txt"
scope, data = read_screen_data(scope, ip_address, 1, 0.1, filename)
t, v_sq, v_ref = data[0]
plotter_single(t, v_sq, v_ref)


In [ ]:
filename = "task4_shapiro_12ghz.txt"
scope, data = read_screen_data(scope, ip_address, 1, 0.1, filename)
t, v_sq, v_ref = data[0]
plotter_single(t, v_sq, v_ref)


In [ ]:
8*200 / 8.5

# Task 6: Power Dependence (Colormap)

In [ ]:
filename = "task6_shapiro_10ghz_11-7power.txt"
scope, data = read_screen_data(scope, ip_address, 1, 0.1, filename)
t, v_sq, v_ref = data[0]
plotter_single(t, v_sq, v_ref)


In [ ]:
filename = "task6_shapiro_10ghz_7-5power.txt"
scope, data = read_screen_data(scope, ip_address, 1, 0.1, filename)
t, v_sq, v_ref = data[0]
plotter_single(t, v_sq, v_ref)


In [ ]:
filename = "task6_shapiro_10ghz_4-6power.txt"
scope, data = read_screen_data(scope, ip_address, 1, 0.1, filename)
t, v_sq, v_ref = data[0]
plotter_single(t, v_sq, v_ref)


In [ ]:
# 

In [ ]:
num_power_steps = 28
wait_between = 5
filename = "task6_power_sweepus.txt"

scope, power_data = read_screen_data(scope, ip_address, num_power_steps, wait_between, filename)

In [ ]:

plotter_colormap(power_data, cmap="inferno")
plotter_derivative_refined(power_data, sigma=2, cmap="RdBu_r")

results were not great, so we re-did it 

In [ ]:
num_power_steps = 28
wait_between = 4
filename = "task6_power_sweepusNEW.txt"

scope, power_data = read_screen_data(scope, ip_address, num_power_steps, wait_between, filename)

plotter_colormap(power_data, cmap="inferno")
plotter_derivative_refined(power_data, sigma=2, cmap="RdBu_r")